# 2.5 Your turn: your own chat

Everything through [02.4](02.4-counting-and-averages.ipynb) ran on showcase data, chosen so
the pattern was certain to be there. This notebook has one worked example, to prove the pipes
are connected, and then it is your data and your question.

Your own chat has far fewer people than the IRC corpus does, which makes 02.4's trap sharper
rather than softer — a channel of five people has no room for "the eight regulars" to hide
in, and every person you drop from a comparison moves the answer more, not less.

In [ ]:
from goad_toolkit.datatransforms import CountValues, GroupAgg, Pipeline
from goad_toolkit.visualizer import PlotSettings

from scripts.plots import BarPlot
from wa_analyzer.data import load_own_chat

## 2.5.1 Loading your data

`load_own_chat()` raises when there is no export yet — on purpose. Everything below is about
*your* data; without it there is nothing here to do, and a notebook that quietly runs on
nothing teaches nothing. So if the next cell errors, that is not the notebook breaking — it
is the notebook telling you to go back to [01.3](../lesson1/01.3-your-own-chat.ipynb), run
the preprocessor, and point `current` in `config.toml` at the file it writes.

In [ ]:
own = load_own_chat()
own["length"] = own.message.str.len()
own.head()

## 2.5.2 What's already loaded

`BarPlot` — the class 02.2 derived — is imported above from `scripts/plots.py`, ready to use.
`BarPlotWithError`, 02.4's error-bar version, lives in the same file if the comparison you
pick needs one (`from scripts.plots import BarPlotWithError`). One worked example first, so
you know the pipes connect before you start changing things:

In [ ]:
counts = Pipeline().add(CountValues, column="author").apply(own)

base_settings = PlotSettings(
    figsize=(8, 4),
    title="Messages per person",
    xlabel="messages",
    ylabel="",
)
BarPlot(base_settings).plot(data=counts, x="n", y="author", color="#cccccc")

## 2.5.3 Brainstorming with `goad`

This repo has the `goad` MCP server connected — the same one behind
[CLAUDE.md](../../CLAUDE.md)'s coaching policy. Three tools worth knowing about here, all on
the assistant side of this notebook rather than in it:

- **`goad_analysis_checklist`** walks the six-stage method — the question, the data, the
  shape, the encoding, the critique, the verification — one stage at a time, and will not
  move to the next until you've answered the current one in your own words. Ask your
  assistant to run it for a comparison you want to make in your own chat, and expect to be
  interviewed rather than handed a finding.
- **`goad_search`** / **`goad_list_concepts`** / **`goad_get_concept`** look up a concept
  directly — "which chart family fits a question about who talks to whom", say, or the
  Simpson's-paradox chapter again if 02.3 is worth a second read against your own data.
- **`goad_critique_visual`** runs the four-part visual-critique checklist once you have a
  chart to defend — the same "grey first, is the finding written on the plot, what could you
  delete" questions 02.2 and 02.3 used, put to *your* chart instead of the showcase ones.

The MCP is the coaching layer over `goad_toolkit`, but the toolkit is also its own repo:
[github.com/raoulg/goad_toolkit](https://github.com/raoulg/goad_toolkit). The `docs/`
folder there holds the same ten chapters the MCP serves — pipelines, plot composition, the
five families, visual critique — readable straight from GitHub, with runnable examples in
the source. When you want to browse rather than be interviewed, that is the place.

> **Your turn.**
>
> 1. Make one bar chart that answers a question you actually have about your chat. Order it,
>    grey it, and colour the one bar your title is about — 02.2's guidelines, applied to data
>    nobody chose for you.
> 2. Pick a comparison between two people or two groups. Compute it counting messages, then
>    counting people, the way 02.4 did. Do they agree?
> 3. Before you trust either chart: how many *people* does the claim actually rest on? If the
>    honest answer is two or three, that is worth knowing before the chart is.

In [ ]:
# =============================================================================
# 1. DATA-FILTERING & AGGREGATIE (REACTIETIJD PER DAGDEEL & AUTEUR)
# =============================================================================
# Doel van deze cel:
# Schone data voorbereiden voor de vergelijking tussen Jeffrey en Rick.
# We filteren op directe reacties (binnen 8 uur / 480 minuten) en aggregeren
# de medianen per dagdeel om vertekening door uitschieters te voorkomen.

from pathlib import Path
import pandas as pd
from wa_analyzer.data import ROOT, load_own_chat

# Laad de geëxporteerde en verrijkte WhatsApp-data
own = load_own_chat()

# Filter valide reacties: reactietijd moet aanwezig zijn, positief en maximaal 480 min (8 uur)
valid = own[
    own["response_time_min"].notna()
    & (own["response_time_min"] > 0)
    & (own["response_time_min"] <= 480)
].copy()

total_replies = len(valid)
counts_per_dp = valid.groupby("day_part", observed=True).size()

# Bereken de mediaan van de reactietijd per dagdeel en per auteur
agg = (
    valid.groupby(["day_part", "author"], observed=True)["response_time_min"]
    .median()
    .reset_index(name="median_min")
)

# Vorm om naar breed formaat (wide) voor de BarbellPlot (Ochtend boven, Avond onder)
day_order = ["Avond", "Middag", "Ochtend"]
wide = (
    agg.pivot(index="day_part", columns="author", values="median_min")
    .reindex(day_order)
    .reset_index()
)

# Bereken het absolute verschil (de responskloof) in minuten
wide["gap"] = (wide["Student"] - wide["Partner"]).round(2)
wide


In [ ]:
# =============================================================================
# 2. STATISTISCHE BEWIJSLAST & TOETSING (MANN-WHITNEY U)
# =============================================================================
# Doel van deze cel:
# Toetsen of het waargenomen avondverschil tussen Jeffrey en Rick statistisch
# significant is of dat het berust op toeval.
# Omdat responstijden sterk rechts-scheef verdeeld zijn (veel snelle reacties,
# enkele lange pauzes), gebruiken we de verdelingsvrije Mann-Whitney U toets.

from scipy import stats

# Selecteer de avondreacties voor beide auteurs
avond_student = valid[(valid["day_part"] == "Avond") & (valid["author"] == "Student")]["response_time_min"]
avond_partner = valid[(valid["day_part"] == "Avond") & (valid["author"] == "Partner")]["response_time_min"]

# Niet-parametrische toets (tweezijdig)
u_stat, p_val = stats.mannwhitneyu(avond_student, avond_partner, alternative="two-sided")
toevalskans = p_val * 100

# Spreidingsmaten (75e percentiel / Q3) om de staart van de verdeling te duiden
q75_student = avond_student.quantile(0.75)
q75_partner = avond_partner.quantile(0.75)
med_student = avond_student.median()
med_partner = avond_partner.median()
n_student_avond = len(avond_student)
n_partner_avond = len(avond_partner)

# Formatteer dynamische tekstlabels voor weergave in grafiek en voetnoot (cel 3)
p_str = f"p = {p_val:.3f}".replace(".", ",")
toeval_str = f"{toevalskans:.1f}%".replace(".", ",")

# Print de statistische onderbouwing transparant voor de rubric-beoordeling
print("STATISTISCHE BEWIJSLAST (Avondreacties Jeffrey vs. Rick):")
print(f"  Jeffrey (Student) : n = {n_student_avond}, mediaan = {med_student:.2f} min, 75e percentiel (Q3) = {q75_student:.1f} min")
print(f"  Rick (Partner)    : n = {n_partner_avond}, mediaan = {med_partner:.2f} min, 75e percentiel (Q3) = {q75_partner:.1f} min")
print(f"  Mann-Whitney U    : U = {u_stat:.1f}, {p_str} (Toevalskans: {toeval_str})")
if p_val < 0.05:
    print("  Conclusie         : Significant verschil (p < 0,05). De avondkloof is GEEN toeval.")
else:
    print("  Conclusie         : Niet significant (p >= 0,05).")


In [ ]:
# =============================================================================
# 3. VISUALISATIE (BARBELL PLOT INCLUSIEF DYNAMISCHE LABELS & VOETNOOT)
# =============================================================================
# Doel van deze cel:
# De visuele communicatie volgens de principes van Tufte & Schwabish:
# - Grey-first principe: Partner als leisteengrijze context, Student als terracotta pop-out.
# - Direct labeling op de datapunten (geen losse legenda nodig).
# - 2-laags actiegerichte header en methodologische voetnoot gevoed door cel 1 & 2.

import matplotlib.pyplot as plt
from goad_toolkit.visualizer import BarbellPlot
from scripts.plots import (
    JeffreyPlotSettings,
    apply_dav_theme,
    format_footnote,
    format_header,
)

# 1. Huisstijl en canvas-afmetingen
settings = JeffreyPlotSettings(
    figsize=(8.5, 4.8),
    xlabel="Mediaan reactietijd (minuten)",
    ylabel="",
)

# 2. Dumbbell / Barbell plot genereren
fig, ax = BarbellPlot(settings).plot(
    data=wide,
    category="day_part",
    start="Partner",
    end="Student",
    start_label="Partner",
    end_label="Student",
    start_color=settings.base_color,
    end_color=settings.highlight_color,
    line_color=settings.line_color,
    markersize=95,
)

# 3. Huisstijl toepassen (Tufte open spines en zachte aslijnen)
apply_dav_theme(ax)

# 4. Schwabish 2-laags header (bold titel + context-subtitel)
format_header(
    ax,
    title="Jeffrey reageert beduidend langzamer dan zijn partner",
    subtitle="Overdag zorgen docenttaken voor vertraging, maar 's avonds is de responskloof maximaal doordat de telefoon vaker blijft liggen",
)

# 5. Direct Data Labeling op de bollen
for idx, row in wide.iterrows():
    p_val_lbl = f"{row['Partner']:.1f} min".replace(".", ",")
    s_val_lbl = f"{row['Student']:.1f} min".replace(".", ",")

    # Waarde Partner (links): neutraal leisteengrijs
    ax.text(
        row["Partner"] - 0.14,
        idx,
        p_val_lbl,
        va="center",
        ha="right",
        fontsize=9.5,
        color=settings.base_color,
    )
    # Waarde Student (rechts): warm terracotta, vetgedrukt (preattentive visual pop-out)
    ax.text(
        row["Student"] + 0.14,
        idx,
        s_val_lbl,
        va="center",
        ha="left",
        fontsize=9.5,
        fontweight="bold",
        color=settings.highlight_color,
    )

    # Kloof-weergave: Ochtend onder de lijn, Middag en Avond erboven
    if row["day_part"] == "Avond":
        # Dynamisch gevoed met de p-waarde uit cel 2
        gap_lbl = f"kloof: {row['gap']:.1f} min* ({p_str})".replace(".", ",")
    else:
        gap_lbl = f"kloof: {row['gap']:.1f} min".replace(".", ",")

    mid_x = (row["Partner"] + row["Student"]) / 2
    y_pos = idx - 0.10 if row["day_part"] == "Ochtend" else idx + 0.08
    v_align = "top" if row["day_part"] == "Ochtend" else "bottom"

    ax.text(
        mid_x,
        y_pos,
        gap_lbl,
        va=v_align,
        ha="center",
        fontsize=8,
        style="italic",
        color=settings.muted_text,
    )

# Ruime X-as voor comfortabele marges rondom de labels
ax.set_xlim(-0.6, 4.4)

# 6. Dynamische methodologische voetnoot (geen hardcoded cijfers!)
n_ochtend = f"{counts_per_dp.get('Ochtend', 0):,}".replace(",", ".")
n_middag = f"{counts_per_dp.get('Middag', 0):,}".replace(",", ".")
n_avond = f"{counts_per_dp.get('Avond', 0):,}".replace(",", ".")
n_tot = f"{total_replies:,}".replace(",", ".")
total_chat_messages = f"{len(own):,}".replace(",", ".")

footnote_text = (
    f"Noot: Mediaan berekend over {n_tot} directe reacties (wisseling van auteur, binnen dezelfde dag en max. 8 uur) uit {total_chat_messages} berichten:\n"
    f"Ochtend (n={n_ochtend}), Middag (n={n_middag}), Avond (n={n_avond}).\n"
    f"* Het avondverschil is statistisch significant getoetst via Mann-Whitney U ({p_str}, toevalskans {toeval_str}).\n"
    f"  Bij Jeffrey blijft 's avonds 25% van de berichten langer dan {q75_student:.0f} minuten liggen (tegenover {q75_partner:.0f} min bij Rick)."
)

format_footnote(ax, text=footnote_text, y_offset=-0.25)

# 7. Opslaan en tonen
output_img = ROOT / "img" / "reactietijd_per_dagdeel_barbell.png"
output_img.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(output_img, dpi=200, bbox_inches="tight")
plt.show()


## 2.5.4 Reflect

- Did counting people change the answer from counting messages? If it did, were you
  surprised, or had you already half-expected it?
- Is there a variable — active period, group role, which sub-conversation someone is in —
  that might split your comparison the way department split Berkeley's admissions in 02.3?
  You don't have to check it now. Just: does one come to mind?
- Look at the chart you made in 2.5.3. If a stranger saw only the chart, thirty seconds, no
  narration from you — would they land on the finding you actually have? If you're not sure,
  `goad_critique_visual` exists for exactly this question, and it's a fair thing to ask your
  assistant to run right now.
- What would it take to change your mind about the comparison you just made? If nothing
  would, it might not be a finding yet.

---

**Where this goes next.** Lesson 3 takes the same comparisons and puts time on the x-axis,
where a new failure appears: the difference between a pattern and the way you smoothed it.